In [ ]:
import requests

#авторизация на MOEX ISS через basic-аутентификацию
MOEX_LOGIN = 'orlovapolya0211@gmail.com' #логин от moex.com
MOEX_PASSWORD = 'E!3UStXK#wAwLgh'  #пароль от moex.com

session = requests.Session()
avtoriz_otvet = session.get(
    'https://passport.moex.com/authenticate',
    auth=(MOEX_LOGIN, MOEX_PASSWORD))

if avtoriz_otvet.status_code == 200:
    print('Авторизация успешна')
    print(f'Cookie получен: {"MicexPassportCert" in session.cookies}')
else:
    print(f'Ошибка авторизации: {avtoriz_otvet.status_code}')

Авторизация успешна
Cookie получен: True


In [ ]:
#cоздаём конфигурационный файл parametrs.txt
#параметры берём из статьи Tproger
#там написано - у basicConfig() три основных параметра - level, filename, format

parametrs_content = """enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
"""
#enabled=true/false - включение/выключение logging без изменения кода
#level=INFO
#filename=logs/logging_infa - где хранятся наши логи
#Tproger: format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
#убрали funcName потому что у нас нет функций
#%(asctime)s  - дата и время события
#%(levelname)s - уровень: INFO в нашем случае
#%(message)s  - само сообщение
#%(lineno)d - номер строки


with open('parametrs.txt', 'w') as filik:
    filik.write(parametrs_content)

print('создали parametrs.txt с такими параметрами:')
with open('parametrs.txt') as filik:
    print(filik.read())

создали parametrs.txt с такими параметрами:
enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s



In [ ]:
!mkdir -p logs #создали папку, где будет файл с логами

In [ ]:
import logging

#читаем наши настройки из parametrs.txt
with open('parametrs.txt') as filik:
    lines = filik.readlines()

enabled = lines[0].split('=')[1].strip()
level_str = lines[1].split('=')[1].strip()
filename = lines[2].split('=')[1].strip()
log_format = lines[3].split('=')[1].strip()

if enabled == 'true':

    #сбрасываем старые handlers
    #потому что иначе basicConfig игнорируется в Colab
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    #filename - из parametrs.txt
    #из Хабра нашли, что записываем сообщения в файл
    #файл будет хранить данные после завершения программы

    #format - из parametrs.txt
    #Tproger: format с %(asctime)s %(levelname)s %(message)s

    #запись в файл
    #Tproger: 'logging.basicConfig() - основная функция для настройки logging'
    #уровень INFO - вывод данных о фрагментах кода, работающих так как ожидается
    logging.basicConfig(level=logging.INFO, filename=filename, format=log_format, filemode='a')
    #filemode='a' - логи добавляются в конец файла и не стираются при перезапуске

    logging.info('Подключаем logging и собираем данные с MOEX ISS API')
    print(f'Logging работает: уровень={level_str}, файл={filename}')

else:
    print('Logging не работает')

Logging работает: уровень=INFO, файл=logs/logging_infa


In [ ]:
import requests #сначала все запросы писали через requests.get() напрямую, потом переписали через session для логгирования
import pandas as pd

Импортируем две библиотеки:
- **requests** - для отправки HTTP-запросов к API биржи (для колаба без логгирования нужна была)
- **pandas** - для работы с таблицами. Превращаем неудобный JSON в удобный DataFrame

## Запрос 1 - Дневные свечи (candles)

**Цель:** получить историю дневных цен Лукойла за период 2018-01-01 - 2025-12-31.

Это главные данные всего проекта. Без них невозможно посчитать нашу таргетную переменную - изменение цены акции через 24 часа после выхода новости.

Каждая строка в ответе = один торговый день.


In [ ]:
#запрос 1: получаем дневные свечи ЛУКОЙЛ с MOEX (первые 500 строк)
response_LKOH = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/LKOH/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24})

data_LKOH = response_LKOH.json()

#извлекаем данные из json
columns_LKOH = data_LKOH['candles']['columns']
rows_LKOH = data_LKOH['candles']['data']

df_LKOH = pd.DataFrame(rows_LKOH, columns=columns_LKOH)
df_LKOH['ticker'] = 'LKOH'

print(f'Count of rows: {len(df_LKOH)}')
#считаем все строки - мало ли их много, а выводит только 500
df_LKOH


Count of rows: 500


,open,close,high,low,value,volume,begin,end,ticker
0,3340.0,3421.5,3421.5,3340.0,1.346153e+09,395934,2018-01-03 00:00:00,2018-01-03 23:59:59,LKOH
1,3428.0,3501.0,3501.0,3392.5,2.426750e+09,702060,2018-01-04 00:00:00,2018-01-04 23:59:59,LKOH
2,3499.5,3569.5,3569.5,3491.0,2.111285e+09,597634,2018-01-05 00:00:00,2018-01-05 23:59:59,LKOH
3,3575.0,3628.5,3647.5,3575.0,3.087868e+09,852342,2018-01-09 00:00:00,2018-01-09 23:59:59,LKOH
4,3640.0,3641.0,3649.0,3594.5,3.720431e+09,1025327,2018-01-10 00:00:00,2018-01-10 23:59:59,LKOH
...,...,...,...,...,...,...,...,...,...
495,6170.0,6190.0,6212.5,6170.0,4.359929e+09,704074,2019-12-16 00:00:00,2019-12-16 23:59:59,LKOH
496,6193.0,6226.0,6238.5,6187.5,5.452464e+09,876963,2019-12-17 00:00:00,2019-12-17 23:59:59,LKOH
497,6227.0,6250.5,6258.5,6224.5,6.291217e+09,1007116,2019-12-18 00:00:00,2019-12-18 23:59:59,LKOH
498,6090.0,6079.0,6150.0,6048.5,9.881849e+09,1621047,2019-12-19 00:00:00,2019-12-19 23:59:59,LKOH


**Результат:**  Count of rows: 500

Таблица содержит 9 колонок:
-  open  - цена акции Лукойла в начале торгового дня (рублей за акцию)
-  close  - цена в конце торгового дня
-  high  - максимальная цена за день
-  low  - минимальная цена за день
-  value  - суммарный оборот за день в рублях (сколько рублей прошло через все сделки)
-  volume  - количество акций сменивших владельца за день
-  begin  - дата и время начала торгового дня
-  end  - дата и время конца торгового дня
-  ticker  - тикер бумаги (добавили сами, чтобы потом объединить все 7 компаний)


Также из документации (стр. 11) узнали что сервер возвращает данные порциями. Чтобы выгрузить полный объём данных, нужно последовательно делать запросы, увеличивая параметр start на значение pagesize, пока не получим пустой блок данных.



In [ ]:
logging.info('Запрос 1: проверяем есть ли данные после 500 строки (start=500)')
#проверяем есть ли данные ниже 500 строки
response_LKOH = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/LKOH/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 500
      })

data_LKOH = response_LKOH.json()
print(f"Count of rows after 500 rows: {len(data_LKOH['candles']['data'])}")
#строк много - выводим оставшиеся


Count of rows after 500 rows: 500


**Результат:**  Count of rows after 500 rows: 500

HOOORRAY!!))

Ну вот видите?)

Данные не закончились на 500-й строке. Получили ещё 500 строк - это оставшиеся торговые дни, которые не влезли в первый запрос.

Теперь будем проверять, пока нам не выведет 0 значений.

In [ ]:
columns_LKOHnew = data_LKOH['candles']['columns']
rows_LKOHnew = data_LKOH['candles']['data']

df_LKOHnew = pd.DataFrame(rows_LKOHnew, columns=columns_LKOHnew)
df_LKOHnew['ticker'] = 'LKOH'

print(f'Count of rows after 500 rows: {len(df_LKOHnew)}')

#проверяем есть ли данные ниже 1000 строки
logging.info('Запрос 1: проверяем есть ли данные после 1000 строки (start=1000)')
response_LKOH2 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/LKOH/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1000
      })

data_LKOH2 = response_LKOH2.json()
columns_LKOHnew2 = data_LKOH2['candles']['columns']
rows_LKOHnew2 = data_LKOH2['candles']['data']

df_LKOHnew2 = pd.DataFrame(rows_LKOHnew2, columns=columns_LKOHnew2)
df_LKOHnew2['ticker'] = 'LKOH'

print(f'Count of rows after 1000 rows: {len(df_LKOHnew2)}')

#проверяем есть ли данные ниже 1500 строки
logging.info('Запрос 1: проверяем есть ли данные после 1500 строки (start=1500)')
response_LKOH3 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/LKOH/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1500
      })

data_LKOH3 = response_LKOH3.json()
columns_LKOHnew3 = data_LKOH3['candles']['columns']
rows_LKOHnew3 = data_LKOH3['candles']['data']

df_LKOHnew3 = pd.DataFrame(rows_LKOHnew3, columns=columns_LKOHnew3)
df_LKOHnew3['ticker'] = 'LKOH'

print(f'Count of rows after 1500 rows: {len(df_LKOHnew3)}')



Count of rows after 500 rows: 500
Count of rows after 1000 rows: 500
Count of rows after 1500 rows: 500


In [ ]:
#проверяем есть ли данные ниже 2000 строки

logging.info('Запрос 1: проверяем есть ли данные после 2000 строки (start=2000)')
response_LKOH4 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/LKOH/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2000
      })

data_LKOH4 = response_LKOH4.json()
columns_LKOHnew4 = data_LKOH4['candles']['columns']
rows_LKOHnew4 = data_LKOH4['candles']['data']

df_LKOHnew4 = pd.DataFrame(rows_LKOHnew4, columns=columns_LKOHnew4)
df_LKOHnew4['ticker'] = 'LKOH'

print(f'Count of rows after 2000 rows: {len(df_LKOHnew4)}')

Count of rows after 2000 rows: 73


In [ ]:
#проверяем есть ли данные ниже 2073 строки
logging.info('Запрос 1: проверяем есть ли данные после 2073 строки (start=2073)')
response_LKOH5 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/LKOH/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2073
      })

data_LKOH5 = response_LKOH5.json()
columns_LKOHnew5 = data_LKOH5['candles']['columns']
rows_LKOHnew5 = data_LKOH5['candles']['data']

df_LKOHnew5 = pd.DataFrame(rows_LKOHnew5, columns=columns_LKOHnew5)
df_LKOHnew5['ticker'] = 'LKOH'

print(f'Count of rows after 2073 rows: {len(df_LKOHnew5)}')

Count of rows after 2073 rows: 0


In [ ]:
#больше данных нет, поэтому обьединяем датафреймы в один
df_LKOHcandles = pd.concat([df_LKOH, df_LKOHnew, df_LKOHnew2, df_LKOHnew3, df_LKOHnew4], ignore_index=True)
print(f'Total rows: {len(df_LKOHcandles)}')

logging.info(f'Запрос 1: итого после склейки всех частей пяти датафреймов - {len(df_LKOHcandles)} строк')
df_LKOHcandles

Total rows: 2073


,open,close,high,low,value,volume,begin,end,ticker
0,3340.0,3421.5,3421.5,3340.0,1.346153e+09,395934,2018-01-03 00:00:00,2018-01-03 23:59:59,LKOH
1,3428.0,3501.0,3501.0,3392.5,2.426750e+09,702060,2018-01-04 00:00:00,2018-01-04 23:59:59,LKOH
2,3499.5,3569.5,3569.5,3491.0,2.111285e+09,597634,2018-01-05 00:00:00,2018-01-05 23:59:59,LKOH
3,3575.0,3628.5,3647.5,3575.0,3.087868e+09,852342,2018-01-09 00:00:00,2018-01-09 23:59:59,LKOH
4,3640.0,3641.0,3649.0,3594.5,3.720431e+09,1025327,2018-01-10 00:00:00,2018-01-10 23:59:59,LKOH
...,...,...,...,...,...,...,...,...,...
2068,5670.0,5790.0,5799.0,5670.0,4.719711e+09,821292,2025-12-26 00:00:00,2025-12-26 23:59:58,LKOH
2069,5792.5,5832.0,5847.0,5791.5,7.206716e+08,123654,2025-12-27 00:00:00,2025-12-27 23:59:57,LKOH
2070,5833.0,5833.0,5838.0,5791.5,6.022870e+08,103554,2025-12-28 00:00:00,2025-12-28 23:59:57,LKOH
2071,5852.0,5832.5,5974.0,5775.0,8.502395e+09,1446155,2025-12-29 00:00:00,2025-12-29 23:59:59,LKOH


Hooray!

Пустой ответ подтверждает, что мы выгрузили абсолютно все данные. После 2073 строк данных нет. Значит итоговый датафрейм  df_LKOHcandles  содержит полную историю дневных котировок Лукойла за весь нужный период без пропусков.

**Результат:**
   
Total rows: 2073
   

Склеили пять кусков через  pd.concat  с параметром  ignore_index=True  - пересчитаем индексы строк заново от 0, иначе в итоговой таблице было бы пять наборов индексов, что вызвало бы у нас путаницу и отсутствие hoorray(.

Итого получили **2073 строк** - это все торговые дни Лукойла за период с 01.01.2018 по 31.12.2025.



## Запрос 2 - Информация о бумаге (security info)

**Цель:** получить полную информацию по акциям Лукойла: официальное название компании, международный код бумаги (ISIN), номинальная стоимость акции и уровень листинга.

Эти данные нужны нам для двух вещей:
1. Красиво описать датасет в README - чтобы было понятно с какими именно бумагами работаем
2. Добавить уровень листинга как признак в EDA - бумаги первого уровня листинга торгуются активнее и имеют более строгие требования к раскрытию информации

In [ ]:
#запрос 2: получаем статические характеристики бумаги ЛУКОЙЛ
#нам возвращается: полное название, ISIN, номинал, уровень листинга, тип бумаги
response_LKOH_infa = session.get(
    'https://iss.moex.com/iss/securities/LKOH.json')

data_LKOH_infa = response_LKOH_infa.json()

#раздел 'description' содержит характеристики в виде строк name/value
columns_LKOH_infa = data_LKOH_infa['description']['columns']
rows_LKOH_infa = data_LKOH_infa['description']['data']

df_LKOH_infa = pd.DataFrame(rows_LKOH_infa, columns=columns_LKOH_infa)
df_LKOH_infa['ticker'] = 'LKOH'

print(f'Count of rows: {len(df_LKOH_infa)}')
df_LKOH_infa

Count of rows: 27


,name,title,value,type,sort_order,is_hidden,precision,ticker
0,SECID,Код ценной бумаги,LKOH,string,1,0,NaN,LKOH
1,ISSUENAME,Наименование ценной бумаги,Акции обыкновенные,string,2,0,NaN,LKOH
2,NAME,Полное наименование,НК ЛУКОЙЛ (ПАО) - ао,string,3,0,NaN,LKOH
3,SHORTNAME,Краткое наименование,ЛУКОЙЛ,string,4,0,NaN,LKOH
4,ISIN,ISIN код,RU0009024277,string,5,0,NaN,LKOH
5,REGNUMBER,Номер государственной регистрации,1-01-00077-A,string,6,0,NaN,LKOH
6,ISSUESIZE,Объем выпуска,692865762,number,7,0,0.0,LKOH
7,FACEVALUE,Номинальная стоимость,0.025,number,8,0,3.0,LKOH
8,FACEUNIT,Валюта номинала,SUR,string,9,0,NaN,LKOH
9,ISSUEDATE,Дата начала торгов,2003-08-20,date,10,0,NaN,LKOH


**Результат:**  Count of rows: 27

Получили 27 строк.

Каждая строка это одна характеристика бумаги.

Каждая характеристика на отдельной строке.

Это неудобно для анализа, поэтому в следующей ячейке мы преобразуем это формат, где одна строка на компанию, каждая характеристика отдельная колонка.

In [ ]:
#преобразуем в удобный формат: одна строка - один тикер
#разворачиваем колонку 'name' в отдельные колонки через pivot
df_LKOH_infa_best = df_LKOH_infa.set_index('name')['value'].to_frame().T
df_LKOH_infa_best['ticker'] = 'LKOH'
df_LKOH_infa_best = df_LKOH_infa_best.reset_index()

print(f'Полное название: {df_LKOH_infa_best["NAME"].values[0]}')
print(f'ISIN: {df_LKOH_infa_best["ISIN"].values[0]}')
print(f'Уровень листинга: {df_LKOH_infa_best["LISTLEVEL"].values[0]}')

logging.info(f'Запрос 2: ISIN={df_LKOH_infa_best["ISIN"].values[0]}, листинг={df_LKOH_infa_best["LISTLEVEL"].values[0]}')

df_LKOH_infa_best

Полное название: НК ЛУКОЙЛ (ПАО) - ао
ISIN: RU0009024277
Уровень листинга: 1


name,index,SECID,ISSUENAME,NAME,SHORTNAME,ISIN,REGNUMBER,ISSUESIZE,FACEVALUE,FACEUNIT,...,MORNINGSESSION,EVENINGSESSION,WEEKENDSESSION,REGISTRY_DATE,TYPENAME,GROUP,TYPE,GROUPNAME,EMITTER_ID,ticker
0,value,LKOH,Акции обыкновенные,НК ЛУКОЙЛ (ПАО) - ао,ЛУКОЙЛ,RU0009024277,1-01-00077-A,692865762,0.025,SUR,...,1,1,1,2003-06-25,Акция обыкновенная,stock_shares,common_share,Акции,770,LKOH


**Результат:**
   
Полное название: НК ЛУКОЙЛ (ПАО) - ао
ISIN: RU0009024277
Уровень листинга: 1
   

Теперь это одна строка с колонками NAME, ISIN, LISTLEVEL и тд.

Что значат выводы:
- **NAME = «НК ЛУКОЙЛ (ПАО) - ао»** - официальное юридическое название компании.
- **ISIN = RU0009024277** - международный идентификационный номер ценной бумаги.
- **LISTLEVEL = 1** - первый уровень листинга на Мосбирже. Это значит, что Лукойл прошёл самые строгие требования биржи по прозрачности и раскрытию информации.

**Итог запроса 2:** получили официальную информацию по бумаге.

Hooray!))

## Запрос 3 - История дивидендов (dividends)

**Цель:** получить все даты дивидендных выплат Лукойла за период проекта.

Зачем нам это нужно: когда компания выплачивает дивиденды, цена акции резко падает примерно на размер дивиденда.

Если мы не пометим эти дни, то можем ошибочно решить в EDA, что резкое падение цены вызвано негативной новостью - хотя на самом деле это просто плановая выплата. Это могло бы нам помешать.

In [ ]:
logging.info('Запрос 3: загружаем историю дивидендов LKOH')
#запрос 3: получаем историю дивидендных выплат ЛУКОЙЛ
#дивиденды влияют на котировки примерно на размер дивиденда
LKOH_diva = session.get(
    'https://iss.moex.com/iss/securities/LKOH/dividends.json')

LKOH_diva_infa = LKOH_diva.json()

#извлекаем данные из json
columns_LKOH_diva = LKOH_diva_infa['dividends']['columns']
rows_LKOH_diva = LKOH_diva_infa['dividends']['data']

df_LKOH_diva = pd.DataFrame(rows_LKOH_diva, columns=columns_LKOH_diva)
df_LKOH_diva['ticker'] = 'LKOH'

print(f'Count of rows: {len(df_LKOH_diva)}')

logging.info(f'Запрос 3: получено {len(df_LKOH_diva)} дивидендных выплат')
df_LKOH_diva

Count of rows: 25


,secid,isin,registryclosedate,value,currencyid,ticker
0,LKOH,RU0009024277,2013-08-15,50,RUB,LKOH
1,LKOH,RU0009024277,2014-07-15,60,RUB,LKOH
2,LKOH,RU0009024277,2014-12-26,60,RUB,LKOH
3,LKOH,RU0009024277,2015-04-28,94,RUB,LKOH
4,LKOH,RU0009024277,2015-07-14,94,RUB,LKOH
5,LKOH,RU0009024277,2015-07-17,94,RUB,LKOH
6,LKOH,RU0009024277,2015-12-24,65,RUB,LKOH
7,LKOH,RU0009024277,2016-07-12,112,RUB,LKOH
8,LKOH,RU0009024277,2016-12-23,75,RUB,LKOH
9,LKOH,RU0009024277,2017-07-10,120,RUB,LKOH


**Результат:**  Count of rows: 25

API вернул 25 выплат дивидентов за все годы существования на бирже. Таблица содержит колонки:

- **registryclosedate** - дата закрытия реестра.
- **value** - размер дивиденда на одну акцию в рублях.

Нам нужны только выплаты за наш период 2018–2025, поэтому сейчас будет фильтровать.

In [ ]:
#фильтруем дивиденды по диапазону дат проекта 2018-01-01 - 2025-12-31
df_LKOH_diva['registryclosedate'] = pd.to_datetime(df_LKOH_diva['registryclosedate'])
df_LKOH_diva_srok = df_LKOH_diva[
    (df_LKOH_diva['registryclosedate'] >= '2018-01-01') &
    (df_LKOH_diva['registryclosedate'] <= '2025-12-31')].reset_index()

print(f'Все дивиденты за 2018-01-01 - 2025-12-31: {len(df_LKOH_diva_srok)}')

logging.info(f'Запрос 2: ISIN={df_LKOH_infa_best["ISIN"].values[0]}, листинг={df_LKOH_infa_best["LISTLEVEL"].values[0]}')
df_LKOH_diva_srok

Все дивиденты за 2018-01-01 - 2025-12-31: 14


,index,secid,isin,registryclosedate,value,currencyid,ticker
0,11,LKOH,RU0009024277,2018-07-11,130,RUB,LKOH
1,12,LKOH,RU0009024277,2018-12-21,95,RUB,LKOH
2,13,LKOH,RU0009024277,2019-07-09,155,RUB,LKOH
3,14,LKOH,RU0009024277,2019-12-20,192,RUB,LKOH
4,15,LKOH,RU0009024277,2020-07-10,350,RUB,LKOH
5,16,LKOH,RU0009024277,2020-12-18,46,RUB,LKOH
6,17,LKOH,RU0009024277,2021-07-05,213,RUB,LKOH
7,18,LKOH,RU0009024277,2021-12-21,340,RUB,LKOH
8,19,LKOH,RU0009024277,2022-12-21,537,RUB,LKOH
9,20,LKOH,RU0009024277,2023-06-05,438,RUB,LKOH


**Результат:**  Все дивиденты за 2018-01-01 - 2025-12-31: 14


Почему конвертировали дату через  pd.to_datetime : без этого Python сравнивал бы строки а не даты. Строковое сравнение работает по алфавиту и даёт неверные результаты для дат в формате YYYY-MM-DD.

**Итог запроса 3:** знаем точные даты дивидендных выплат. В EDA пометим эти дни флагом  is_dividend_day=1 .

## Запрос 4 - Текущие рыночные данные (market data)

Запрос 4 получает текущие торговые данные по Лукойлу - последнюю цену, объём торгов за день, количество сделок и капитализацию компании. Всё это показывает насколько активно торгуется акция прямо сейчас.


Эти данные характеризуют **ликвидность** бумаги - насколько активно она торгуется. В EDA мы будем использовать их чтобы показать: более ликвидные бумаги быстрее и точнее реагируют на новости, потому что больше участников рынка мгновенно перекладываются после публикации статьи.

In [ ]:
logging.info('Запрос 4: загружаем текущие рыночные данные LKOH')
#запрос 4: получаем текущие торговые данные ЛУКОЙЛ
response_LKOH_markdata = session.get(
    'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/LKOH.json')

data_LKOH_markdata = response_LKOH_markdata.json()


#раздел 'marketdata' содержит торговые данные текущей/последней сессии
columns_LKOH_markdata = data_LKOH_markdata['marketdata']['columns']
rows_LKOH_markdata = data_LKOH_markdata['marketdata']['data']

df_LKOH_markdata = pd.DataFrame(rows_LKOH_markdata, columns=columns_LKOH_markdata)
df_LKOH_markdata['ticker'] = 'LKOH'

print(f'Count of rows: {len(df_LKOH_markdata)}')

logging.info(f'Запрос 4: получено {len(df_LKOH_markdata)} строк market data')
df_LKOH_markdata

Count of rows: 1


,SECID,BOARDID,BID,BIDDEPTH,OFFER,OFFERDEPTH,SPREAD,BIDDEPTHT,OFFERDEPTHT,OPEN,...,SYSTIME,CLOSINGAUCTIONPRICE,CLOSINGAUCTIONVOLUME,ISSUECAPITALIZATION,ISSUECAPITALIZATION_UPDATETIME,ETFSETTLECURRENCY,VALTODAY_RUR,TRADINGSESSION,TRENDISSUECAPITALIZATION,ticker
0,LKOH,TQBR,5783.5,None,5784,None,0.5,70596,213284,5830.5,...,2026-03-19 22:39:10,5812,1532,4017582120957,22:38:56,None,10332859907,2,-13510882359,LKOH


**Результат:**  Count of rows: 1

Одна строка - потому что это текущие торговые данные. Но колонок очень много. API возвращает абсолютно всё что знает о торговле этой бумагой прямо сейчас.

Большинство колонок нам не нужны - это технические поля биржи вроде кодов режимов торгов, флагов состояния и прочего. В следующей ячейке оставим только то что важно для нашего анализа.

In [ ]:
#оставляем только нужные колонки из market data
#используем фильтрацию через список - так код не упадёт если какой-то колонки нет в ответе
markdata_cols = ['ticker', 'LAST', 'LASTTOPREVPRICE', 'BID', 'OFFER',
            'OPEN', 'HIGH', 'LOW', 'WAPRICE', 'NUMTRADES',
            'VOLTODAY', 'VALTODAY', 'ISSUECAPITALIZATION', 'UPDATETIME']
markdata_cols = [col for col in markdata_cols if col in df_LKOH_markdata.columns]

df_LKOH_markdata_best = df_LKOH_markdata[markdata_cols].copy()

#выводим ключевые показатели
print(f'Последняя цена LKOH: {df_LKOH_markdata_best["LAST"].values[0]}')
print(f'Изменение к пред. закрытию: {df_LKOH_markdata_best["LASTTOPREVPRICE"].values[0]}')
print(f'Капитализация: {df_LKOH_markdata_best["ISSUECAPITALIZATION"].values[0]}')

logging.info(f'Запрос 4: последняя цена LKOH={df_LKOH_markdata_best["LAST"].values[0]}')

df_LKOH_markdata_best

Последняя цена LKOH: 5783.5
Изменение к пред. закрытию: -0.59
Капитализация: 4017582120957


,ticker,LAST,LASTTOPREVPRICE,BID,OFFER,OPEN,HIGH,LOW,WAPRICE,NUMTRADES,VOLTODAY,VALTODAY,ISSUECAPITALIZATION,UPDATETIME
0,LKOH,5783.5,-0.59,5783.5,5784,5830.5,5874,5780.5,5830.5,74393,1772240,10332859907,4017582120957,22:24:10


**Результат:** (тут данные могут изменяться при каждом запуске)
   
Последняя цена LKOH: 5785.5
Изменение к пред. закрытию: -0.56
Капитализация: 4013771359266
   

Что означают колонки которые мы оставили:
- **LAST** - последняя цена сделки (рублей за акцию)
- **LASTTOPREVPRICE** - изменение в % к цене закрытия предыдущего дня
- **BID / OFFER** - лучшая цена покупки и продажи прямо сейчас. Разница между ними называется спредом - чем он уже, тем ликвиднее бумага
- **WAPRICE** - средневзвешенная цена за весь день (учитывает объём каждой сделки)
- **NUMTRADES** - количество сделок за торговый день. У Лукойла это десятки тысяч - одна из самых торгуемых бумаг на бирже
- **VOLTODAY / VALTODAY** - объём в акциях и оборот в рублях за сегодня
- **ISSUECAPITALIZATION** - рыночная капитализация в рублях.

**Итог запроса 4:** получили характеристику ликвидности Лукойла на текущий момент.

## Запрос 5 - Индексная принадлежность (indices)

**Цель:** узнать в какие биржевые индексы входит Лукойл.

Индексные бумаги - особая категория. Когда акция входит в IMOEX (Индекс МосБиржи), за ней автоматически следят индексные фонды (ETF) и управляющие компании. Это означает что при выходе важной новости реагируют не только частные инвесторы, но и алгоритмы фондов.

В EDA мы сравним: одинаково ли сильно реагируют на новости индексные бумаги и менее крупные компании из нашего списка.

In [ ]:
logging.info('Запрос 5: загружаем индексы LKOH')
#запрос 5: получаем список биржевых индексов в которые входит ЛУКОЙЛ
response_LKOH_idishka = session.get(
    'https://iss.moex.com/iss/securities/LKOH/indices.json')

data_LKOH_idishka = response_LKOH_idishka.json()

#извлекаем данные из json
columns_LKOH_idishka = data_LKOH_idishka['indices']['columns']
rows_LKOH_idishka = data_LKOH_idishka['indices']['data']

df_LKOH_idishka = pd.DataFrame(rows_LKOH_idishka, columns=columns_LKOH_idishka)
df_LKOH_idishka['ticker'] = 'LKOH'

print(f'Count of rows: {len(df_LKOH_idishka)}')

logging.info(f'Запрос 5: LKOH входит в {len(df_LKOH_idishka)} индексов')

df_LKOH_idishka

Count of rows: 21


,SECID,SHORTNAME,FROM,TILL,CURRENTVALUE,LASTCHANGEPRC,LASTCHANGE,ticker
0,EPSI,Субиндекс акций,2007-12-28,2026-03-18,1665.38,0.00,-0.02,LKOH
1,HCAP,HCAP,2024-06-21,2024-09-26,907.67,-0.07,-0.66,LKOH
2,ICLIMATE,Индекс МосБиржи Климатический,2025-07-01,2026-03-18,1030.83,0.23,2.39,LKOH
3,IMOEXCNY,Индекс МосБиржи в юанях,2024-09-24,2026-03-18,1060.38,-2.49,-27.11,LKOH
4,IMOEXW,IMOEXW,2023-01-16,2026-03-18,2881.58,-0.15,-4.34,LKOH
5,MESG,Индекс МосБиржи-RAEX ESG,2023-02-27,2026-03-18,984.75,-0.27,-2.69,LKOH
6,MOEXBC,Индекс голубых фишек,2010-01-14,2026-03-19,19057.56,-0.03,-5.43,LKOH
7,MRRT,Индекс MRRT,2020-09-21,2026-03-18,2335.80,0.05,1.15,LKOH
8,MRSV,Индекс MRSV,2020-09-21,2026-03-18,2173.86,-0.22,-4.80,LKOH
9,MRSVR,Индекс МосБиржи-РСПП MRSV RU Co,2021-03-25,2026-03-18,2237.45,-0.22,-4.93,LKOH


**Результат:**  Count of rows: 21


Важные колонки:
- **SECID** - код индекса
- **FROM / TILL** - период с которого по который бумага входит в индекс
- **CURRENTVALUE** - текущее значение самого индекса

In [ ]:
#проверяем входит ли LKOH в IMOEX (главный российский фондовый индекс)
if len(df_LKOH_idishka) > 0:
    imoex_check = df_LKOH_idishka[df_LKOH_idishka['SECID'] == 'IMOEX']
    print(f'LKOH входит в IMOEX: {len(imoex_check) > 0}')
else:
    print('LKOH не входит ни в один индекс MOEX')
logging.info(f'Запрос 5 завершили - все данные по LKOH собраны')


LKOH входит в IMOEX: True


**Результат:**
   
LKOH входит в IMOEX: True
   

Лукойл входит в IMOEX - это подтверждает что он является влиятельной компанией российского рынка.

**Итог запроса 5:** Лукойл входит в IMOEX и другие индексы Мосбиржи. В EDA это будет важным признаком при анализе силы реакции на новости.



